# LeWM Push-T: Prescribed vs Free CNN (Pixels)
**Runtime → Change runtime type → T4 GPU**, потом запусти ячейку ниже.

Всё в одной ячейке. ~20-30 мин.

In [ ]:
# ============================================================
# SETUP
# ============================================================
!pip install h5py huggingface_hub hdf5plugin -q

import os, time, json, zipfile
import numpy as np
import h5py
import hdf5plugin
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ============================================================
# DOWNLOAD DATA
# ============================================================
H5_PATH = '/content/pusht_expert_train.h5'

if not os.path.exists(H5_PATH):
    from huggingface_hub import hf_hub_download
    print('Downloading 13 GB...')
    zst_path = hf_hub_download('quentinll/lewm-pusht',
        'pusht_expert_train.h5.zst', repo_type='dataset', local_dir='/content/')
    print(f'Downloaded: {zst_path}')
    !apt-get install -y zstd -qq
    print('Decompressing...')
    !zstd -d "{zst_path}" -o {H5_PATH} && rm -f "{zst_path}"
    # Clean up zst and cache
    !rm -rf /root/.cache/huggingface
    print('Done!')
else:
    print('Data exists')

!ls -lh {H5_PATH}

# Check structure
with h5py.File(H5_PATH, 'r') as f:
    print('Keys:', list(f.keys()))
    for k in f.keys():
        if isinstance(f[k], h5py.Dataset):
            print(f'  {k}: {f[k].shape} {f[k].dtype}')
    print(f'State[0]: {f["state"][0]}')

# ============================================================
# DATASET (lazy, reads from disk)
# ============================================================
H = 3
FRAMESKIP = 5
MAX_EP = 200  # keep small for speed
SEED = 42

class LazyDS(Dataset):
    def __init__(self, h5_path, H=3, fs=5, max_ep=200):
        self.path = h5_path
        self.H = H
        self.windows = []
        with h5py.File(h5_path, 'r') as f:
            ep_len = f['ep_len'][:]
            ep_off = f['ep_offset'][:]
            n = min(len(ep_len), max_ep)
            for i in range(n):
                off, ln = int(ep_off[i]), int(ep_len[i])
                idx = list(range(off, off+ln, fs))
                for t in range(len(idx) - H - 1):
                    self.windows.append(idx[t:t+H+2])
                if (i+1)%50==0: print(f'  {i+1}/{n} ep, {len(self.windows)} win')
        print(f'Dataset: {len(self.windows)} windows from {n} episodes')

    def __len__(self): return len(self.windows)

    def __getitem__(self, i):
        idx = self.windows[i]
        with h5py.File(self.path, 'r') as f:
            pix = f['pixels'][idx]       # (H+2, 224, 224, 3)
            state = f['state'][idx]      # (H+2, 7)
            act = f['action'][idx[:-1]]  # (H+1, 2)
        pix = torch.from_numpy(pix).float().permute(0,3,1,2) / 255.0
        pix = F.interpolate(pix, size=96, mode='bilinear', align_corners=False)
        return pix, torch.from_numpy(act).float(), torch.from_numpy(state).float()

ds = LazyDS(H5_PATH, H, FRAMESKIP, MAX_EP)

# Quick test
pix, act, st = ds[0]
print(f'pix={pix.shape} act={act.shape} st={st.shape}')
print(f'block: x={st[0,2]:.1f} y={st[0,3]:.1f} angle={st[0,4]:.3f}')

# Split
torch.manual_seed(SEED)
nt = int(len(ds)*0.9); nv = len(ds)-nt
tr, va = random_split(ds, [nt,nv], generator=torch.Generator().manual_seed(SEED))
train_dl = DataLoader(tr, batch_size=32, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(va, batch_size=32, num_workers=2, pin_memory=True)
print(f'Train={nt} Val={nv}')

# ============================================================
# MODELS
# ============================================================
class SIGReg(nn.Module):
    def __init__(self, knots=17, np_=512):
        super().__init__()
        self.np = np_
        t = torch.linspace(0,3,knots); dt=3/(knots-1)
        w = torch.full((knots,),2*dt); w[[0,-1]]=dt
        phi = torch.exp(-t.square()/2)
        self.register_buffer('t',t)
        self.register_buffer('phi',phi)
        self.register_buffer('w',w*phi)
    def forward(self, p):
        A=torch.randn(p.size(-1),self.np,device=p.device)
        A=A.div_(A.norm(p=2,dim=0))
        x=(p@A).unsqueeze(-1)*self.t
        e=(x.cos().mean(-3)-self.phi).square()+x.sin().mean(-3).square()
        return ((e@self.w)*p.size(-2)).mean()

class CNN(nn.Module):
    def __init__(self, D=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,32,4,2,1), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32,64,4,2,1), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64,128,4,2,1), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128,256,4,2,1), nn.BatchNorm2d(256), nn.GELU(),
            nn.AdaptiveAvgPool2d(1))
        self.proj = nn.Sequential(nn.Linear(256,128),nn.GELU(),nn.Linear(128,D))
    def forward(self, pix, st=None):
        B,T = pix.shape[:2]
        x = self.conv(pix.reshape(B*T,*pix.shape[2:])).squeeze(-1).squeeze(-1)
        return self.proj(x).reshape(B,T,-1)

class PrescEnc(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('sc',torch.tensor([1/512,1/512,1/6.2832]))
    def forward(self, pix, st): return st[...,2:5]*self.sc

class ActEnc(nn.Module):
    def __init__(self,D=3):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(2,32),nn.GELU(),nn.Linear(32,D))
    def forward(self,a): return self.net(a)

class Pred(nn.Module):
    def __init__(self,D=3,H=3):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(H*2*D,128),nn.LayerNorm(128),nn.GELU(),
            nn.Linear(128,128),nn.LayerNorm(128),nn.GELU(),
            nn.Linear(128,D))
    def forward(self,e,ae):
        return self.net(torch.cat([e,ae],dim=-1).reshape(e.size(0),-1))

# ============================================================
# TRAINING
# ============================================================
def run(mode, train_dl, val_dl, D=3, H=3, epochs=15, lam=0.09, seed=42):
    torch.manual_seed(seed)
    enc = PrescEnc().to(device) if mode=='prescribed' else CNN(D).to(device)
    aenc = ActEnc(D).to(device)
    pred = Pred(D,H).to(device)
    sig = SIGReg().to(device)
    params = list(enc.parameters())+list(aenc.parameters())+list(pred.parameters())
    np_ = sum(p.numel() for p in params if p.requires_grad)
    opt = torch.optim.AdamW([p for p in params if p.requires_grad], lr=3e-4, weight_decay=1e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    print(f'\n--- {mode} ({np_:,} params) ---')

    hist, best, bep = [], float('inf'), 0
    for ep in range(1, epochs+1):
        t0 = time.time()
        enc.train(); aenc.train(); pred.train()
        tp,ts,n = 0,0,0
        for pix,act,state in train_dl:
            pix,act,state = pix.to(device),act.to(device),state.to(device)
            emb = enc(pix,state)
            ctx,tgt = emb[:,:H],emb[:,H]
            ae = aenc(act[:,:H])
            p = pred(ctx,ae)
            pl = F.mse_loss(p,tgt.detach())
            sl = sig(emb.transpose(0,1))
            loss = pl + lam*sl
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(params,1.0); opt.step()
            b=pix.size(0); tp+=pl.item()*b; ts+=sl.item()*b; n+=b
        tp/=n; ts/=n

        enc.eval(); aenc.eval(); pred.eval()
        vp,vn = 0,0
        with torch.no_grad():
            for pix,act,state in val_dl:
                pix,act,state = pix.to(device),act.to(device),state.to(device)
                emb=enc(pix,state); ctx,tgt=emb[:,:H],emb[:,H]
                ae=aenc(act[:,:H]); p=pred(ctx,ae)
                vp+=F.mse_loss(p,tgt).item()*pix.size(0); vn+=pix.size(0)
        vp/=vn
        if vp<best: best,bep=vp,ep
        sch.step()
        hist.append({'ep':ep,'tp':tp,'vp':vp,'ts':ts})
        print(f'  ep {ep:2d} | train {tp:.6f} | val {vp:.6f} | sig {ts:.4f} | {time.time()-t0:.1f}s')
    print(f'  BEST: {best:.6f} (ep {bep})')
    return {'mode':mode,'params':np_,'best_vp':best,'best_ep':bep,'history':hist}

# ============================================================
# RUN
# ============================================================
results = {}
for mode in ['prescribed', 'free_cnn']:
    results[mode] = run(mode, train_dl, val_dl, D=3, H=H, epochs=15, seed=SEED)

# ============================================================
# RESULTS
# ============================================================
print('\n' + '='*60)
print('RESULTS — PIXELS (LeWM Push-T HDF5)'.center(60))
print('='*60)
for m in ['prescribed','free_cnn']:
    r=results[m]
    print(f"  {m:15s}: best={r['best_vp']:.6f} ep={r['best_ep']} params={r['params']:,}")
p,f = results['prescribed']['best_vp'], results['free_cnn']['best_vp']
ratio = f/p if p>0 else 0
print(f'\n  Ratio free/prescribed = {ratio:.1f}x')
print(f'  {"PRESCRIBED лучше" if p<f else "FREE лучше"}')

# Plot
import matplotlib.pyplot as plt
fig,ax = plt.subplots(figsize=(8,5))
for m,c in [('prescribed','#2ecc71'),('free_cnn','#e74c3c')]:
    h=results[m]['history']
    ax.plot([x['ep'] for x in h],[x['vp'] for x in h],
            label=f"{m} ({results[m]['params']:,}p)",color=c,lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Val pred_loss')
ax.set_yscale('log'); ax.legend(); ax.grid(True,alpha=0.3)
ax.set_title('LeWM Push-T Pixels: Prescribed vs Free CNN')
plt.tight_layout()
plt.savefig('lewm_pixels.png',dpi=150,bbox_inches='tight')
plt.show()

# Save & download
out = {'results':{k:{kk:vv for kk,vv in v.items()} for k,v in results.items()}}
with open('lewm_pixels.json','w') as fh: json.dump(out,fh,indent=2)
with zipfile.ZipFile('lewm_pixels_all.zip','w') as z:
    z.write('lewm_pixels.json'); z.write('lewm_pixels.png')
from google.colab import files
files.download('lewm_pixels_all.zip')
print('\nDONE!')